# Clase 7: Ajuste fino eficiente

Tomamos un modelo base de 6.000 millones de parámetros, que solo continúa texto, y lo convertimos en un asistente que responde en castellano. Todo en la GPU gratuita de Colab.

**Pregunta de la clase:** ¿cómo se ajusta un modelo que, entero, no cabe en la GPU?

**Objetivos de la clase**

- Cargar un modelo en 4 bits y medir cuánto ocupa.
- Preparar pares de instrucción y respuesta para el ajuste fino supervisado (SFT).
- Entrenar LoRA sobre el modelo cuantizado (QLoRA) y comparar el antes y el después.

Modelo: BERTIN GPT-J-6B (Apache 2.0). Datos: Aya Dataset (Apache 2.0).

## 0. Preparación

Este notebook necesita una GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

La celda siguiente instala tres bibliotecas: `bitsandbytes` cuantiza, `peft` agrega LoRA y `trl` entrena con pares de instrucción y respuesta. Después descarga el modelo, unos 24 GB. Conviene correrla al empezar la clase.

In [ ]:
!pip install -q bitsandbytes==0.50.2 peft==0.21.2 trl==1.17.1

import torch
from huggingface_hub import snapshot_download

MODELO = "bertin-project/bertin-gpt-j-6B"

print("GPU:", torch.cuda.get_device_name(0))
memoria_libre, memoria_total = torch.cuda.mem_get_info()
print(f"Memoria de la GPU: {memoria_total / 1e9:.1f} GB")

# el repositorio trae los pesos dos veces (safetensors y .bin, 24 GB cada
# uno): bajamos una sola copia, más la configuración y el tokenizador
carpeta = snapshot_download(
    MODELO, allow_patterns=["*.json", "*.txt", "model.safetensors"])
print("Modelo descargado en", carpeta)

## 1. ¿Cuánto ocupa el modelo?

El modelo es BERTIN GPT-J-6B. GPT-J-6B lo publicó EleutherAI en 2021, entrenado con texto en inglés. En 2022, el proyecto BERTIN lo siguió entrenando con unos 65.000 millones de tokens en castellano. Es un modelo base: aprendió a predecir el token siguiente y nada más.

Contemos sus parámetros sin cargar los pesos. Dentro de `init_empty_weights`, el modelo se arma vacío, solo con la forma de cada matriz.

In [ ]:
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM

configuracion_modelo = AutoConfig.from_pretrained(MODELO)
with init_empty_weights():
    modelo_vacio = AutoModelForCausalLM.from_config(configuracion_modelo)

parametros = sum(p.numel() for p in modelo_vacio.parameters())
print(f"Parámetros: {parametros:,}")
print("Bloques:", configuracion_modelo.n_layer)
print("Dimensión de los embeddings:", configuracion_modelo.n_embd)

Memoria = parámetros × bytes por número. Antes de correr la celda: ¿en qué formato cabe el modelo en esta GPU? ¿Y cuánto haría falta para entrenarlo entero con Adam?

In [ ]:
bytes_por_parametro = {
    "fp32": 4,
    "fp16": 2,
    "4 bits": 0.5,
    # como en el deck: pesos (2), gradientes (2), copia en fp32 (4) y Adam (8)
    "entrenar todo con Adam": 16,
}
for formato, n_bytes in bytes_por_parametro.items():
    print(f"{formato:>24}: {parametros * n_bytes / 1e9:5.1f} GB")
print(f"{'memoria libre de la GPU':>24}: {memoria_libre / 1e9:5.1f} GB")

Carguémoslo en 4 bits, con las opciones de QLoRA:

- `bnb_4bit_quant_type="nf4"`: los 16 niveles de NF4, más juntos cerca de cero.
- `bnb_4bit_use_double_quant=True`: la doble cuantización. Las escalas de cada bloque también se guardan con menos bits.
- `bnb_4bit_compute_dtype=torch.float16`: para multiplicar, cada bloque se descuantiza a fp16. La T4 no calcula en bf16.

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig

cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO,
    quantization_config=cuantizacion,
    device_map={"": 0},  # todo el modelo en la GPU 0
    # los pesos que no se cuantizan quedan en fp16
    dtype=torch.float16,
)
tokenizador = AutoTokenizer.from_pretrained(MODELO)
# GPT-J no trae token de padding: usamos el de fin
tokenizador.pad_token = tokenizador.eos_token

print(f"En la GPU ocupa {modelo.get_memory_footprint() / 1e9:.1f} GB")

¿Ocupa justo medio byte por parámetro? Sumemos los bytes según el tipo de cada parámetro.

In [ ]:
from collections import Counter

bytes_por_tipo = Counter()
for parametro in modelo.parameters():
    bytes_por_tipo[str(parametro.dtype)] += (
        parametro.numel() * parametro.element_size())

for tipo, n_bytes in bytes_por_tipo.items():
    print(f"{tipo:>14}: {n_bytes / 1e9:.2f} GB")

Los pesos en 4 bits se guardan de a dos por byte, como enteros de 8 bits (`uint8`). Lo que queda en fp16 son los embeddings, la capa de salida y las normalizaciones: bitsandbytes cuantiza las capas lineales de los bloques.

¿Cuánto cambió cada peso? Leamos del archivo los primeros seis pesos de una capa y comparémoslos con los mismos pesos después de pasar a 4 bits y volver, como en la tabla de absmax del deck.

In [ ]:
import bitsandbytes.functional as bnb_funciones
from safetensors import safe_open

# h.0 es el primer bloque del modelo
nombre = "transformer.h.0.mlp.fc_in.weight"
with safe_open(f"{carpeta}/model.safetensors", framework="pt") as archivo:
    originales = archivo.get_tensor(nombre)

capa = modelo.transformer.h[0].mlp.fc_in
de_vuelta = bnb_funciones.dequantize_4bit(
    capa.weight.data, capa.weight.quant_state)
# a fp32 y a la CPU, para compararlos con los originales
de_vuelta = de_vuelta.float().cpu()

# [0, :6]: los seis primeros pesos de la primera fila
print("originales:", originales[0, :6].numpy().round(4))
print("de vuelta: ", de_vuelta[0, :6].numpy().round(4))

error_medio = (de_vuelta - originales).abs().mean()
tamano_medio = originales.abs().mean()
print(f"Error medio en la capa: {error_medio / tamano_medio:.0%} "
      "del tamaño medio de un peso")

¿Sigue funcionando el modelo con todos sus pesos así?

## 2. El antes

Le damos al modelo una instrucción con el formato de Alpaca del deck: «### Instrucción:», la instrucción y «### Respuesta:». El modelo genera hasta 200 tokens, eligiendo siempre el más probable (decodificación greedy), y se detiene si produce el token de fin, `<|endoftext|>`.

In [ ]:
FORMATO = "### Instrucción:\n{instruccion}\n\n### Respuesta:\n"


@torch.no_grad()
def responder(modelo, instruccion: str) -> tuple[str, bool]:
    # devuelve la respuesta y si el modelo se detuvo solo
    modelo.eval()
    prompt = FORMATO.format(instruccion=instruccion)
    entrada = tokenizador(prompt, return_tensors="pt").to(0)
    salida = modelo.generate(
        **entrada,
        max_new_tokens=200,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizador.eos_token_id,
        pad_token_id=tokenizador.eos_token_id,
    )
    # salida trae el prompt seguido de los tokens nuevos
    largo_prompt = entrada.input_ids.shape[1]
    tokens_generados = salida[0, largo_prompt:]
    se_detuvo = tokens_generados[-1].item() == tokenizador.eos_token_id
    respuesta = tokenizador.decode(
        tokens_generados, skip_special_tokens=True)
    return respuesta, se_detuvo

Cinco instrucciones fijas y una del chat. Cambien `pregunta_del_chat` por la que propongan.

In [ ]:
INSTRUCCIONES = [
    "Escribe un correo breve para pedir una reunión el lunes.",
    "¿Cuál es la capital de Chile?",
    "Explica en dos frases qué es la fotosíntesis.",
    "Dame tres ideas para un almuerzo vegetariano barato.",
    "Traduce al inglés: «Mañana va a llover en Santiago».",
]
# cambien esta pregunta por la que propongan en el chat
pregunta_del_chat = "¿Qué me recomiendas visitar en Valparaíso?"

`probar_instrucciones` le da al modelo las seis instrucciones, muestra los primeros 500 caracteres de cada respuesta y cuenta cuántas se detuvieron solas. La usamos ahora, con el modelo base, y en la sección 6, con el modelo ajustado.

In [ ]:
def probar_instrucciones(modelo) -> None:
    detenidas = 0
    todas = INSTRUCCIONES + [pregunta_del_chat]
    index = 0
    for instruccion in todas:
        index += 1
        print(f"EJEMPLO {index}")
        respuesta, se_detuvo = responder(modelo, instruccion)
        detenidas += se_detuvo
        estado = "se detuvo" if se_detuvo else "llegó a 200 tokens"
        print(f"--- {instruccion} [{estado}]")
        print(respuesta[:500], "\n")
    print(f"Se detuvieron {detenidas} de {len(todas)}")

Antes de correr: ¿cuántas de las seis respuestas se van a detener solas?

In [ ]:
probar_instrucciones(modelo)

## 3. Los datos

Usemos 1.000 pares de instrucción y respuesta en castellano, el tamaño de LIMA. Vienen del Aya Dataset (Singh et al., 2024): unos 204.000 pares en 65 idiomas, escritos por personas.

In [ ]:
from datasets import load_dataset

aya = load_dataset("CohereLabs/aya_dataset", split="train")
aya_castellano = aya.filter(lambda fila: fila["language_code"] == "spa")
# con la semilla fija, siempre salen los mismos pares
aya_castellano = aya_castellano.shuffle(seed=0)
print("Pares en castellano:", len(aya_castellano))

Cada par queda en dos columnas: `prompt`, la instrucción con el formato, y `completion`, la respuesta. Con esas dos columnas, `trl` sabe qué parte cuenta para la pérdida. Dejamos fuera los pares de 512 tokens o más.

In [ ]:
from datasets import Dataset

pares = []
for fila in aya_castellano:
    prompt = FORMATO.format(instruccion=fila["inputs"].strip())
    respuesta = fila["targets"].strip()
    # con el token de fin, estos pares no cabrían en max_length=512
    if len(tokenizador(prompt + respuesta).input_ids) >= 512:
        continue
    pares.append({"prompt": prompt, "completion": respuesta})
    if len(pares) == 1000:
        break
pares = Dataset.from_list(pares)

for numero, par in enumerate(pares.select(range(2)), start=1):
    print(f"PAR {numero}:")
    print(par["prompt"] + par["completion"], "\n")

## 4. LoRA y el entrenador

La configuración de LoRA es la del deck: r = 16 y α = 32, así que la corrección se multiplica por α/r = 2. LoRA va en todas las capas lineales, como en QLoRA. `lora_dropout=0.05` es dropout, el que ya conocen, aplicado solo al camino de LoRA: en cada paso de entrenamiento, cada valor de la entrada x se pone en cero con probabilidad 5 % antes de pasar por A, mientras W recibe la entrada completa. Al generar no se aplica.

In [ ]:
from peft import LoraConfig

configuracion_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

El entrenamiento: una epoch sobre los 1.000 pares, en lotes de 4 con acumulación de gradientes de 4, así que cada paso ve 16 pares. Learning rate de 2 × 10⁻⁴, que baja en forma de coseno. fp16, porque la T4 no tiene bf16, y gradient checkpointing.

In [ ]:
from trl import SFTConfig

configuracion_entrenamiento = SFTConfig(
    output_dir="ajuste",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    # el learning rate sube durante los tres primeros pasos
    warmup_steps=3,
    max_length=512,
    # cada secuencia lleva un solo par
    packing=False,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

`prepare_model_for_kbit_training` deja listo el modelo cuantizado: congela sus pesos, pasa las normalizaciones a fp32 y activa el gradient checkpointing.

In [ ]:
from peft import prepare_model_for_kbit_training

modelo = prepare_model_for_kbit_training(modelo)
# la caché solo acelera la generación, y no se usa con gradient checkpointing
modelo.config.use_cache = False

Con `peft_config`, el `SFTTrainer` agrega un par A y B a cada capa lineal. También prepara los pares: los tokeniza y marca qué tokens cuentan para la pérdida.

In [ ]:
from trl import SFTTrainer

entrenador = SFTTrainer(
    model=modelo,
    args=configuracion_entrenamiento,
    train_dataset=pares,
    processing_class=tokenizador,
    peft_config=configuracion_lora,
)

Con un modelo cuantizado, esta versión de `trl` deja A y B en bf16, y el escalado de gradientes de `fp16=True` no acepta ese formato. Los pasamos a fp32: es la copia en fp32 de la precisión mixta que vimos en el deck.

In [ ]:
# solo A y B tienen requires_grad: el resto del modelo está congelado
for parametro in entrenador.model.parameters():
    if parametro.requires_grad:
        parametro.data = parametro.data.float()

¿Cuántos parámetros se van a entrenar, y en qué capas?

In [ ]:
entrenables = 0
tipos_de_capa_con_lora = set()
for nombre, parametro in entrenador.model.named_parameters():
    if parametro.requires_grad:
        entrenables += parametro.numel()
    if "lora_A" in nombre:
        # el nombre termina, por ejemplo, en
        # «h.0.attn.q_proj.lora_A.default.weight»: contando desde el
        # final, [-4] es el tipo de capa, q_proj, sin el número de bloque
        partes_del_nombre = nombre.split(".")
        tipos_de_capa_con_lora.add(partes_del_nombre[-4])

print(f"Parámetros entrenables: {entrenables:,}")
print(f"Fracción del modelo: {entrenables / parametros:.2%}")
# A y B están en fp32: peso (4 bytes), gradiente (4) y Adam (8);
# la cuenta deja fuera el modelo base y las activaciones
print(f"Pesos, gradientes y Adam, a 16 bytes por parámetro: "
      f"{entrenables * 16 / 1e9:.1f} GB")
print("Capas con LoRA:", sorted(tipos_de_capa_con_lora))

¿Qué cuenta para la pérdida? Pidamos al entrenador un lote con el primer par: los tokens con etiqueta −100 no cuentan.

In [ ]:
lote = entrenador.data_collator([entrenador.train_dataset[0]])
# el lote tiene forma [pares, tokens]; [0] toma el único par
ids = lote["input_ids"][0]
etiquetas = lote["labels"][0]
mascara_respuesta = etiquetas != -100

print("No cuentan:")
print(tokenizador.decode(ids[~mascara_respuesta]))
print("\nCuentan:")
print(tokenizador.decode(ids[mascara_respuesta]))
print(f"\n{mascara_respuesta.sum()} de {len(ids)} tokens "
      "cuentan para la pérdida")

La instrucción entra como contexto, pero solo la respuesta y el token de fin cuentan para la pérdida. Ese token es el que le enseña al modelo a parar.

## 5. Entrenar

Una epoch, 63 pasos. Lo lanzamos y nos vamos al recreo.

In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
inicio = time.time()
entrenador.train()

minutos = (time.time() - inicio) / 60
# la memoria que PyTorch tomó de la GPU, aunque no toda esté en uso
pico_memoria_gb = torch.cuda.max_memory_reserved() / 1e9
print(f"Entrenamiento: {minutos:.1f} minutos")
print(f"Pico de memoria de la GPU: {pico_memoria_gb:.1f} GB")

La pérdida, cada 5 pasos:

In [ ]:
import matplotlib.pyplot as plt

registros = [r for r in entrenador.state.log_history if "loss" in r]
plt.figure(figsize=(6, 3))
plt.plot([r["step"] for r in registros], [r["loss"] for r in registros],
         marker="o")
plt.xlabel("paso")
plt.ylabel("pérdida")
plt.show()

## 6. El después

Corramos `probar_instrucciones`, de la sección 2, con el modelo ajustado: las mismas seis instrucciones y el mismo formato. ¿Aprendió a parar?

In [ ]:
probar_instrucciones(entrenador.model)

## 7. Cierre

Lo que se entrenó cabe en un archivo aparte, el adapter: solo las matrices A y B.

In [ ]:
import os

entrenador.model.save_pretrained("adapter")
tamano_adapter = sum(os.path.getsize(os.path.join("adapter", archivo))
                     for archivo in os.listdir("adapter"))
tamano_modelo = os.path.getsize(f"{carpeta}/model.safetensors")
print(f"Adapter: {tamano_adapter / 1e6:.0f} MB")
print(f"Modelo en fp32: {tamano_modelo / 1e9:.1f} GB")

Para usarlo otro día se carga el modelo base, en 4 bits como acá, y se le pone el adapter encima:

```python
from peft import PeftModel

modelo_ajustado = PeftModel.from_pretrained(modelo_base, "adapter")
```

¿Cómo se ajusta un modelo que, entero, no cabe en la GPU? Los pesos del modelo se guardan en 4 bits y no se entrenan. Lo que se entrena es LoRA, las matrices A y B, con sus gradientes y Adam.

La clase 6 mostró que un buen prompt resuelve mucho sin entrenar. Ajustar conviene cuando el prompt no alcanza y hay ejemplos de lo que se quiere: un formato fijo, un estilo, una tarea con datos propios. Con QLoRA, ese ajuste cabe en una GPU gratuita.

**Lecturas**

- Hu, E. J. et al. (2021). «LoRA: Low-Rank Adaptation of Large Language Models». https://arxiv.org/abs/2106.09685
- Dettmers, T., Pagnoni, A., Holtzman, A. y Zettlemoyer, L. (2023). «QLoRA: Efficient Finetuning of Quantized LLMs». https://arxiv.org/abs/2305.14314
- Zhou, C. et al. (2023). «LIMA: Less Is More for Alignment». https://arxiv.org/abs/2305.11206
- Singh, S. et al. (2024). «Aya Dataset: An Open-Access Collection for Multilingual Instruction Tuning». https://arxiv.org/abs/2402.06619
- Hugging Face. Documentación de PEFT (https://huggingface.co/docs/peft) y del `SFTTrainer` de TRL (https://huggingface.co/docs/trl/sft_trainer).

## Ejercicio propuesto

¿Cuántos ejemplos hacen falta? Entrenen LoRA desde cero con 100 pares en vez de 1.000 y prueben las mismas instrucciones con `probar_instrucciones`. ¿Cuántas respuestas se detienen? ¿Cómo son? ¿Y con 300?

In [ ]:
# cambien a 300 para repetir el ejercicio con más pares
n_pares = 100

# unload() quita A y B: queda el modelo base en 4 bits
modelo_base = entrenador.model.unload()
del entrenador
torch.cuda.empty_cache()

entrenador_chico = SFTTrainer(
    model=modelo_base,
    args=configuracion_entrenamiento,
    train_dataset=pares.select(range(n_pares)),
    processing_class=tokenizador,
    peft_config=configuracion_lora,
)
# como antes, A y B a fp32 para el escalado de gradientes de fp16
for parametro in entrenador_chico.model.parameters():
    if parametro.requires_grad:
        parametro.data = parametro.data.float()

entrenador_chico.train()
probar_instrucciones(entrenador_chico.model)